# CoMarketer — Test Deployed Endpoint
Run this notebook from Databricks to test the deployed app.

In [1]:
# Config
APP_URL = "https://comarketer-2276245144129479.aws.databricksapps.com"

## 1. Get Auth Token
Uses the Databricks notebook context token if running on Databricks,
otherwise reads from your local `.databrickscfg`.

In [2]:
import os

def get_token():
    """Get Databricks PAT token from notebook context or env."""
    # If running on Databricks
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        return w.config.token
    except Exception:
        pass
    # If running locally
    token = os.environ.get("DATABRICKS_TOKEN")
    if token:
        return token
    # Read from .databrickscfg
    import configparser
    cfg = configparser.ConfigParser()
    cfg.read(os.path.expanduser("~/.databrickscfg"))
    return cfg.get("DEFAULT", "token", fallback=None)

TOKEN = get_token()
print("Token found:", "Yes" if TOKEN else "No — set DATABRICKS_TOKEN env var")

Token found: Yes


## 2. Test `/invocations` — Known Client (IGP)

In [4]:
import requests
import json

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {TOKEN}",
}

payload = {
    "input": [{"role": "user", "content": "Hello! How are my campaigns doing?"}],
    "custom_inputs": {"sp_id": "sp-igp-001"},
}

resp = requests.post(f"{APP_URL}/invocations", headers=headers, json=payload)
print(f"Status: {resp.status_code}")
print(f"Response:\n{json.dumps(resp.json(), indent=2)}")

Status: 200


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## 3. Validate Response Structure

In [ ]:
data = resp.json()

# Check output structure
assert "output" in data, "Missing 'output' key"
assert len(data["output"]) > 0, "Empty output"

message = data["output"][0]
assert message["role"] == "assistant", f"Expected role=assistant, got {message['role']}"

text = message["content"][0]["text"]
assert "IGP" in text, f"Expected 'IGP' in response, got: {text[:100]}"

# Check custom_outputs
custom = data.get("custom_outputs", {})
assert custom.get("client_id") == "72994", f"Expected client_id=72994, got {custom.get('client_id')}"
assert custom.get("client_name") == "IGP", f"Expected client_name=IGP, got {custom.get('client_name')}"
assert "request_id" in custom, "Missing request_id"
assert "trace_id" in custom, "Missing trace_id"

print("\n--- Results ---")
print(f"Client: {custom['client_name']} (ID: {custom['client_id']})")
print(f"Request ID: {custom['request_id']}")
print(f"Trace ID: {custom['trace_id']}")
print(f"Response preview: {text[:120]}...")
print("\n✅ Test 1 PASSED: Known client (IGP) returns correct response")

## 4. Test `/invocations` — Pepe Jeans Client

In [ ]:
payload_pepe = {
    "input": [{"role": "user", "content": "Show me last week's email stats"}],
    "custom_inputs": {"sp_id": "sp-pepe-002"},
}

resp2 = requests.post(f"{APP_URL}/invocations", headers=headers, json=payload_pepe)
data2 = resp2.json()

text2 = data2["output"][0]["content"][0]["text"]
custom2 = data2.get("custom_outputs", {})

assert "Pepe Jeans" in text2, f"Expected 'Pepe Jeans' in response"
assert custom2.get("client_id") == "81001"

print(f"Client: {custom2['client_name']} (ID: {custom2['client_id']})")
print(f"Response preview: {text2[:120]}...")
print("\n✅ Test 2 PASSED: Pepe Jeans client returns correct response")

## 5. Test `/invocations` — Unknown SP (fallback to Demo)

In [ ]:
payload_unknown = {
    "input": [{"role": "user", "content": "Hi there"}],
    "custom_inputs": {"sp_id": "unknown-sp-xyz"},
}

resp3 = requests.post(f"{APP_URL}/invocations", headers=headers, json=payload_unknown)
data3 = resp3.json()

custom3 = data3.get("custom_outputs", {})
assert custom3.get("client_name") == "Demo", f"Expected Demo fallback, got {custom3.get('client_name')}"
assert custom3.get("client_id") == "00000"

print(f"Client: {custom3['client_name']} (ID: {custom3['client_id']})")
print("\n✅ Test 3 PASSED: Unknown SP falls back to Demo client")

## 6. Test `/ui/health` Endpoint

In [ ]:
resp_health = requests.get(f"{APP_URL}/ui/health", headers=headers)
health = resp_health.json()

assert health["status"] == "healthy"
assert health["agent"] == "comarketer"
assert health["mode"] == "dummy"

print(f"Health: {health}")
print("\n✅ Test 4 PASSED: Health endpoint returns correct status")

## 7. Summary

In [ ]:
print("="*50)
print("  CoMarketer Endpoint Test Summary")
print("="*50)
print("✅ Test 1: Known client (IGP) — PASSED")
print("✅ Test 2: Known client (Pepe Jeans) — PASSED")
print("✅ Test 3: Unknown SP fallback — PASSED")
print("✅ Test 4: Health endpoint — PASSED")
print("="*50)
print(f"\nApp URL: {APP_URL}")
print(f"UI: {APP_URL}/ui/")
print(f"API: {APP_URL}/invocations")
print(f"Health: {APP_URL}/ui/health")